### 5.1 Evaluating generative text models

In [9]:
import torch
import torch.nn as nn
import tiktoken
from previous_chapters import GPTModel, generate_text_simple

#### 5.1.1 Using GPT to generate text

In [6]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [8]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();

In [10]:
def text_to_token_ids(text: str, tokenizer) -> torch.tensor:    
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    return torch.tensor(encoded).unsqueeze(0)   # batch dimehsion added

def token_ids_to_text(token_ids: torch.tensor, tokenizer) -> str:
    ids_list =  token_ids.squeeze(0).tolist()  # batch dim removed
    return tokenizer.decode(ids_list)

In [11]:
start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

In [12]:
out = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M['context_length']
)

out

tensor([[ 6109,  3626,  6100,   345, 34245,  5139,  2492, 25405, 17434, 17853,
          5308,  3398, 13174, 43071]])

In [13]:
token_ids_to_text(out, tokenizer)

'Every effort moves you rentingetic wasnم refres RexMeCHicular stren'

#### 5.1.2 Calculating the text generation loss

In [16]:
inputs = torch.tensor(
    [[16833, 3626, 6100], # ["every effort moves",
     [40, 1107, 588]])     # "I really like"]

targets = torch.tensor(
    [[3626, 6100, 345 ],  # [" effort moves you",
     [1107, 588, 11311]]) # " really like chocolate"]

In [18]:
with torch.no_grad():
    logits = model(inputs)

probas = torch.softmax(logits, dim=-1)
print(probas.shape)
probas

torch.Size([2, 3, 50257])


tensor([[[1.8849e-05, 1.5172e-05, 1.1687e-05,  ..., 2.2409e-05,
          6.9776e-06, 1.8776e-05],
         [9.1569e-06, 1.0062e-05, 7.8786e-06,  ..., 2.9090e-05,
          6.0103e-06, 1.3571e-05],
         [2.9877e-05, 8.8507e-06, 1.5741e-05,  ..., 3.5456e-05,
          1.4094e-05, 1.3526e-05]],

        [[1.2561e-05, 2.0538e-05, 1.4332e-05,  ..., 1.0389e-05,
          3.4784e-05, 1.4239e-05],
         [7.2731e-06, 1.7864e-05, 1.0565e-05,  ..., 2.1206e-05,
          1.1390e-05, 1.5559e-05],
         [2.9496e-05, 3.3605e-05, 4.1029e-05,  ..., 6.5249e-06,
          5.8203e-05, 1.3698e-05]]])

In [20]:
idx = torch.argmax(probas, dim=-1, keepdim=True)
print(idx.shape)
idx

torch.Size([2, 3, 1])


tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])

In [28]:
print('Target:', token_ids_to_text(targets[0], tokenizer))
print('Output:', token_ids_to_text(idx[0].flatten(), tokenizer))

Target:  effort moves you
Output:  Armed heNetflix


In [33]:
print(probas[0].shape)
probas[0]

torch.Size([3, 50257])


tensor([[1.8849e-05, 1.5172e-05, 1.1687e-05,  ..., 2.2409e-05, 6.9776e-06,
         1.8776e-05],
        [9.1569e-06, 1.0062e-05, 7.8786e-06,  ..., 2.9090e-05, 6.0103e-06,
         1.3571e-05],
        [2.9877e-05, 8.8507e-06, 1.5741e-05,  ..., 3.5456e-05, 1.4094e-05,
         1.3526e-05]])

In [45]:
# Probabilites of target token IDS ([3626, 6100, 345]) of the first text
text_idx = 0
target_probas_1 = probas[text_idx,[0,1,2],targets[text_idx]]
target_probas_1

tensor([7.4540e-05, 3.1061e-05, 1.1563e-05])

In [46]:
# Probabilites of target token IDS ([1107, 588, 11311]) of the second text
text_idx = 1
target_probas_2 = probas[text_idx,[0,1,2],targets[text_idx]]
target_probas_2

tensor([1.0337e-05, 5.6776e-05, 4.7559e-06])

In [49]:
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
log_probas

tensor([ -9.5042, -10.3796, -11.3677, -11.4798,  -9.7764, -12.2561])

In [ ]:
# CROSS ENTROPY LOSS: Negative average Log probability
neg_avg_log_proba = -torch.mean(log_probas)
neg_avg_log_proba

tensor(10.7940)

In [51]:
logits.shape, targets.shape

(torch.Size([2, 3, 50257]), torch.Size([2, 3]))

In [53]:
logits

tensor([[[ 0.1113, -0.1057, -0.3666,  ...,  0.2843, -0.8824,  0.1074],
         [-0.6109, -0.5167, -0.7613,  ...,  0.5450, -1.0319, -0.2175],
         [ 0.5707, -0.6459, -0.0701,  ...,  0.7419, -0.1806, -0.2217]],

        [[-0.2968,  0.1949, -0.1649,  ..., -0.4867,  0.7218, -0.1714],
         [-0.8375,  0.0612, -0.4641,  ...,  0.2327, -0.3889, -0.0770],
         [ 0.5614,  0.6919,  0.8915,  ..., -0.9472,  1.2411, -0.2056]]])

In [61]:
logits_flat = logits.flatten(0,1)
print(logits_flat.shape)
logits_flat

torch.Size([6, 50257])


tensor([[ 0.1113, -0.1057, -0.3666,  ...,  0.2843, -0.8824,  0.1074],
        [-0.6109, -0.5167, -0.7613,  ...,  0.5450, -1.0319, -0.2175],
        [ 0.5707, -0.6459, -0.0701,  ...,  0.7419, -0.1806, -0.2217],
        [-0.2968,  0.1949, -0.1649,  ..., -0.4867,  0.7218, -0.1714],
        [-0.8375,  0.0612, -0.4641,  ...,  0.2327, -0.3889, -0.0770],
        [ 0.5614,  0.6919,  0.8915,  ..., -0.9472,  1.2411, -0.2056]])

In [55]:
targets

tensor([[ 3626,  6100,   345],
        [ 1107,   588, 11311]])

In [60]:
target_flat = targets.flatten()
target_flat.shape, target_flat

(torch.Size([6]), tensor([ 3626,  6100,   345,  1107,   588, 11311]))

In [62]:
loss = nn.functional.cross_entropy(logits_flat, target_flat)
loss

tensor(10.7940)

In [64]:
perplexity = torch.exp(loss)
perplexity

tensor(48725.8203)

#### 5.1.3 Calculating the training and validation set losses

In [ ]:
import os
import shutil

In [ ]:
colab = True

In [ ]:
if colab:
    from google.colab import drive
    
    if os.path.exists("/content/drive"):
        print("Google Drive has already mounted.")
    else:        
        drive.mount('/content/drive')
        print("Google Drive is mounted")

    shutil.copytree("./drive/MyDrive/LLM_from_scratch/", "./", dirs_exist_ok=True);

Mounted at /content/drive
Google Drive is mounted


In [3]:

# copy files to the Colab server
# ! cp ./drive/MyDrive/LLM_from_scratch/* .

In [4]:
import torch
import torch.nn as nn
import tiktoken

import previous_chapters as prv

In [5]:
print(torch.__version__)
torch.cuda.is_available()

2.10.0+cu128


True

In [6]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

tokenizer = tiktoken.get_encoding('gpt2')

In [7]:
torch.manual_seed(123)
model = prv.GPTModel(GPT_CONFIG_124M)
model.eval();

In [8]:
filepath = 'the-verdict.txt' if colab else '../../data/the-verdict.txt'

with open(filepath,'r', encoding='utf-8') as f:
    text_data = f.read()

tokens = tokenizer.encode(text_data)

len(text_data), len(tokens)

(20479, 5145)

In [9]:
train_ratio = 0.9

split_idx = int(len(text_data) * train_ratio)
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

train_data[-10:], val_data[:10]

("y 'techniq", "ue' collap")

In [10]:
torch.manual_seed(123)

train_dataset = prv.create_dataloader_v1(
    txt=train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M['context_length'],
    stride=GPT_CONFIG_124M['context_length'],
    shuffle=True,
    drop_last=True
)

val_dataset = prv.create_dataloader_v1(
    txt=val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M['context_length'],
    stride=GPT_CONFIG_124M['context_length'],
    shuffle=True,
    drop_last=True
)

In [11]:
for i, (x,y) in enumerate(train_dataset, 1):
    print(f'{i}. X shape: {x.shape}, Y shape: {y.shape}')

1. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])
2. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])
3. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])
4. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])
5. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])
6. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])
7. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])
8. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])
9. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])


In [12]:
for i, (x,y) in enumerate(val_dataset, 1):
    print(f'{i}. X shape: {x.shape}, Y shape: {y.shape}')

1. X shape: torch.Size([2, 256]), Y shape: torch.Size([2, 256])


In [13]:
train_iter = iter(train_dataset)
x, y = next(train_iter)

In [14]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits_batch = model(input_batch)

    loss = nn.functional.cross_entropy(
        logits_batch.flatten(0,1),
        target_batch.flatten()
        )
    return loss

In [15]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    if len(data_loader) == 0:
        return float("nan")
    
    num_batches = min(num_batches, len(data_loader)) if num_batches else len(data_loader)

    total_loss = 0
    for i, (x_batch, y_batch) in enumerate(data_loader):
        if i < num_batches:
            batch_loss = calc_loss_batch(x_batch, y_batch, model, device)
            total_loss += batch_loss
        else:
            break

    return total_loss / num_batches

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

with torch.no_grad():
    train_loss = calc_loss_loader(train_dataset, model, device)
    val_loss = calc_loss_loader(val_dataset, model, device)

In [17]:
print(f'Train Loss: {train_loss}')
print(f'Val Loss: {val_loss}')

Train Loss: 10.98758316040039
Val Loss: 10.98110580444336
